# AutoTriage — train on Kaggle

Runs the whole training pipeline for the [AutoTriage](https://github.com/dhyanagni2001-commits/autotriage-workflow) project:

1. Data prep (Bitext customer-support dataset -> classifier splits + retrieval pairs + synthetic KB)
2. Bi-encoder retriever fine-tuning + FAISS index build + eval vs. off-the-shelf baseline
3. DistilBERT classifier fine-tuning + eval vs. TF-IDF baseline
4. Packages everything into `autotriage_artifacts.zip` under Notebook Output for download

**Before running:** in the Kaggle notebook settings (right sidebar), turn on:
- **Accelerator**: GPU (T4 x2 or P100)
- **Internet**: On (needed to `pip install`, pull the HF dataset, and optionally `git clone` the repo)

## Resuming after a timeout / disconnect

Both training scripts (`retriever/train_biencoder.py`, `classifier/train_classifier.py`) checkpoint
every epoch and auto-resume from the latest checkpoint when re-run — they will **not** restart
from scratch. To carry that state across a Kaggle *session* (a session's `/kaggle/working` is
wiped on restart unless you save it):

1. After a run (complete or interrupted), click **Save Version** so `/kaggle/working` becomes
   this notebook's Output, i.e. a Kaggle Dataset you can attach elsewhere.
2. Next session: **Add Input** -> your own notebook's previous output (search "Your Work" ->
   this notebook) or a dataset you made from it. Set `RESUME_INPUT_DIR` below to its path.
3. Re-run all cells. The "restore checkpoints" cell copies the prior run's checkpoints back into
   place before training resumes, so you only ever lose the current, incomplete epoch — not the
   whole run.

In [ ]:
import os

# --- configure this per run ---
REPO_URL = "https://github.com/dhyanagni2001-commits/autotriage-workflow.git"
REPO_DIR = "/kaggle/working/autotriage-workflow"

# If you attached a previous run's output (or a dataset made from it) as an
# input, point this at its path, e.g. "/kaggle/input/autotriage-checkpoints".
# Leave as None for a fresh run.
RESUME_INPUT_DIR = None

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 1. Get the repo

Tries `git clone` first (works once you've pushed your local changes to GitHub). If that fails
(private repo, no internet, or you haven't pushed yet), it falls back to an input dataset named
`autotriage-workflow` that you've uploaded manually — zip your local repo and add it as a Kaggle
Dataset input if you go that route.

In [ ]:
import shutil
import subprocess
from pathlib import Path

if Path(REPO_DIR).exists():
    print(f"{REPO_DIR} already exists, reusing it.")
else:
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
    if result.returncode != 0:
        uploaded = next(Path("/kaggle/input").glob("*autotriage*"), None)
        if uploaded is None:
            raise SystemExit(
                "git clone failed and no /kaggle/input/*autotriage* dataset was found. "
                "Push the repo to GitHub, or add it as an input dataset, then re-run."
            )
        print(f"git clone failed; copying uploaded dataset from {uploaded} instead.")
        shutil.copytree(uploaded, REPO_DIR)

print("repo ready at", REPO_DIR)
os.chdir(REPO_DIR)

In [ ]:
# Install training deps only — deliberately NOT reinstalling torch, since
# Kaggle's base image already has a CUDA-matched torch build and pulling in
# requirements.txt's torch>=2.2 unpinned could silently replace it with a
# CPU-only or mismatched-CUDA wheel.
!pip install -q "transformers>=4.40" "sentence-transformers>=3.0" "datasets>=2.19" \
    "accelerate>=0.30" "faiss-cpu>=1.8" "scikit-learn>=1.4" "pyyaml>=6.0" "tqdm>=4.66"

## 2. Restore checkpoints from a previous run (if resuming)

In [ ]:
import yaml

cfg = yaml.safe_load(open("config.yaml"))

RESTORE_PATHS = [
    cfg["data"]["processed_dir"],
    cfg["retriever"]["output_dir"],
    str(Path(cfg["retriever"]["faiss_index_path"]).parent),
    cfg["classifier"]["output_dir"],
]

if RESUME_INPUT_DIR and Path(RESUME_INPUT_DIR).exists():
    for rel in RESTORE_PATHS:
        src = Path(RESUME_INPUT_DIR) / rel
        if not src.exists():
            continue
        dst = Path(REPO_DIR) / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"restored {src} -> {dst}")
else:
    print("RESUME_INPUT_DIR not set (or missing) — starting fresh.")

## 3. Data prep

Skips itself if `data/processed/classifier_train.jsonl` already exists (e.g. restored from a
previous run) — data prep is deterministic given the same seed, so there's nothing to gain from
re-running it, and re-running it would also be wasted time on every resume.

In [ ]:
processed_marker = Path(cfg["data"]["processed_dir"]) / "classifier_train.jsonl"
if processed_marker.exists():
    print(f"{processed_marker} already exists — skipping data prep.")
else:
    subprocess.run(["python", "data/prepare_bitext.py", "--config", "config.yaml"], check=True)

## 4. Bi-encoder retriever: train, build index, eval

`train_biencoder.py` checkpoints every epoch and resumes automatically — safe to re-run this cell
after any interruption.

In [ ]:
subprocess.run(["python", "retriever/train_biencoder.py", "--config", "config.yaml"], check=True)

In [ ]:
subprocess.run(["python", "retriever/build_index.py", "--config", "config.yaml"], check=True)

In [ ]:
retrieval_eval = subprocess.run(
    ["python", "retriever/eval_retrieval.py", "--config", "config.yaml"],
    check=True, capture_output=True, text=True,
)
print(retrieval_eval.stdout)

## 5. DistilBERT classifier: train, eval

`train_classifier.py` uses HuggingFace `Trainer`'s built-in checkpointing and resumes from the
latest `checkpoint-*` automatically — also safe to re-run after any interruption. It no-ops if a
`TRAINING_COMPLETE` marker is already present.

In [ ]:
subprocess.run(["python", "classifier/train_classifier.py", "--config", "config.yaml"], check=True)

In [ ]:
classifier_eval = subprocess.run(
    ["python", "classifier/eval_classifier.py", "--config", "config.yaml"],
    check=True, capture_output=True, text=True,
)
print(classifier_eval.stdout)

## 6. Package artifacts for download

Zips the trained checkpoints, FAISS index, processed data, and eval output text into one file
under `/kaggle/working` so it shows up in the notebook's **Output** tab. Download it, unzip into
your local repo at the matching paths (`retriever/checkpoints/...`, `classifier/checkpoints/...`,
`retriever/faiss_index/...`, `data/processed/...`), then paste the results table below into the
main `README.md`.

In [ ]:
results_path = Path("/kaggle/working/eval_results.txt")
results_path.write_text(
    "retriever\n" + retrieval_eval.stdout + "\nclassifier\n" + classifier_eval.stdout
)
print(results_path.read_text())

artifact_dir = Path("/kaggle/working/autotriage_artifacts")
if artifact_dir.exists():
    shutil.rmtree(artifact_dir)
artifact_dir.mkdir(parents=True)

for rel in RESTORE_PATHS:
    src = Path(REPO_DIR) / rel
    if src.exists():
        shutil.copytree(src, artifact_dir / rel)

shutil.copy(results_path, artifact_dir / "eval_results.txt")

zip_path = shutil.make_archive("/kaggle/working/autotriage_artifacts", "zip", artifact_dir)
print("wrote", zip_path)